## Lernziel

Deutsche Tutorial-Notebooks direkt in Quarto aufnehmen, Ergebnisse prüfen und die Website lokal ansehen.

**Voraussetzungen:** Quarto CLI und die Publikationsabhängigkeiten; für vorhandene statische Grafikexporte Chrome/Kaleido.

Einmalig im Repository installieren: `python -m pip install -e ".[publication]"`.
Zum eigenen Ausführen das Notebook im geklonten Repository öffnen und die Zellen von oben nach unten ausführen.
Die Website zeigt gespeicherte Ausgaben; sie führt keinen Python-Code im Browser aus.

[Tutorial-Notebook herunterladen](downloads/tutorial_website.ipynb) · [Alle Tutorials](python.qmd)


## 1. Quellen und Ergebnisse unterscheiden

`website/tutorial_*.ipynb` sind redaktionell gepflegte deutsche Tutorials mit Quarto-Kopf und gespeicherten Ausgaben.
Die ursprünglichen `notebooks/howto_*` bleiben eigenständige Arbeitsnotebooks. Es gibt keine generierte
Notebook-Seitenkopie zwischen diesen beiden Beständen. Für Berechnung, Import und Grafik verwenden beide die Paketfunktionen.

Quarto muss separat installiert sein. Der Website-Build rendert gespeicherte Ergebnisse und führt die Tutorialzellen nicht aus.
Neue oder geänderte Analyse-Tutorials deshalb vorher im passenden Kernel ausführen und speichern.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# Im geklonten Repository ausführen, z. B. aus website/ oder notebooks/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/pexl").is_dir() and (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Bitte das Notebook innerhalb des peexcel-Repositorys öffnen.")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
OUTPUT = ROOT / "notebooks/local_exports/tutorials"
OUTPUT.mkdir(parents=True, exist_ok=True)
import nbformat
import yaml
config = yaml.safe_load((ROOT / "website/_quarto.yml").read_text(encoding="utf-8"))
tutorials = sorted((ROOT / "website").glob("tutorial_*.ipynb"))
for pfad in tutorials:
    nbformat.validate(nbformat.read(pfad, as_version=4))
print("Direkt gerenderte Tutorialquellen:", len(tutorials))
print("Render-Auswahl:", config["project"]["render"])

Direkt gerenderte Tutorialquellen: 12
Render-Auswahl: ['*.qmd', 'tutorial_*.ipynb', '!downloads/**', '!en/**']


## 2. Ein Tutorial ergänzen

Ein vorhandenes `tutorial_*.ipynb` als redaktionelles Muster verwenden: eigenständiges Lernziel,
Voraussetzungen, Datenquelle, Konfiguration, nachvollziehbare Schritte, Ergebnis und Grenzen.
Der rohe Quarto-Kopf enthält `title`, `lang: de` und `execute: {enabled: false}`.

`_quarto.yml` rendert die Tutorialdateien direkt. Ergänze dort den Eintrag in der passenden Sidebar-Gruppe
und auf `python.qmd` eine Zeile mit Ziel und Voraussetzungen. Auch Tutorials zur Excel-Bedienung können
als Notebooks mit Markdown und Abbildungen aufgenommen werden; Python-Code ist keine Pflicht.
Der Notebook-Download wird vom Asset-Schritt unverändert kopiert, einschließlich gespeicherter Ausgaben.

## 3. Ressourcen und Seiten bauen

Der Asset-Schritt bereitet Downloads und bestehende Lehrmodule vor. Danach rendert Quarto.
Die folgenden Schalter bleiben für das normale Durchlaufen dieses Tutorials ausgeschaltet;
so erzeugt die Dokumentation keine rekursiven Website-Builds.
Falls ein Pre-render-Hook aktiv ist, übernimmt dieser den Asset-Schritt; die Konfiguration unten berücksichtigt das.

In [2]:
import subprocess
BUILD_AUSFUEHREN = False
if BUILD_AUSFUEHREN:
    if not config["project"].get("pre-render"):
        subprocess.run([sys.executable, "scripts/website_assets.py"], cwd=ROOT, check=True)
    subprocess.run(["quarto", "render", "website"], cwd=ROOT, check=True)
else:
    print("Build nicht gestartet. Für den bewussten lokalen Build BUILD_AUSFUEHREN = True setzen.")

Build nicht gestartet. Für den bewussten lokalen Build BUILD_AUSFUEHREN = True setzen.


## 4. Vorschau und Abnahme

Nach dem Build im Repository ausführen:

```bash
python -m http.server 8765 --bind 127.0.0.1 --directory website/_site
```

Dann `http://127.0.0.1:8765/python.html` öffnen. Übersicht, Seitenleiste, Suche und Downloads prüfen.
Diagramme müssen sichtbar und bedienbar sein; Code muss aufklappbar bleiben. Die Seite auch bei schmaler Breite prüfen.
Website-Tests mit `python -m pytest tests/test_website.py tests/test_tutorials.py` ausführen.

## 5. Veröffentlichung getrennt durchführen

Ein lokaler Build veröffentlicht nichts. Vor einer Freigabe alle gewünschten Sprachfassungen prüfen:
Das englische Unterprojekt unter `website/en` wird separat gebaut. Der Veröffentlichungszweig ist `gh-pages`.
Die Quarto-Veröffentlichung erfolgt erst mit dem dafür vorgesehenen Auftrag; dieses Notebook startet sie nicht.

## Ergebnis

Die Website rendert die gepflegten Tutorial-Notebooks direkt. Ausgaben stammen aus dokumentierten lokalen Läufen;
Excel- und API-Tutorials dürfen ohne passende Umgebung als noch nicht ausgeführt gekennzeichnet bleiben.



## Weiterarbeiten

[Zur Tutorial-Übersicht](python.qmd). Die ursprünglichen Arbeitsnotebooks bleiben erhalten: [howto_website](https://github.com/simonschaluppe/peexcel/blob/master/notebooks/howto_website.ipynb).
